# Soma detection by scale-normalised Laplacian-of-Gaussian (LoG)

**Why LoG instead of FDT.** The FDT-based soma detector
(`02_fdt_blob_mask.ipynb` -- Pipeline A) measures *local thickness*: it
finds the points deepest inside a bright connected region. It fails
when somas are bright cytoplasmic rings of *similar intensity to
surrounding dendrites* -- the FDT cannot tell a soma from a dense
neurite tangle.

The scale-normalised Laplacian-of-Gaussian (Lindeberg, *IJCV* 30(2):
77-116, 1998) is a **size-tuned spot detector**: it returns local
maxima of `sigma^2 * |LoG(image)|` over a range of scales. Each
detection is `(row, col, sigma)` -- a position plus the scale at which
the spot was strongest. The blob radius is `sqrt(2) * sigma`. Tune the
sigma range to soma radius and LoG finds round cell bodies *directly*,
independent of FDT thickness reasoning.

**Workflow** mirrors the FDT notebook: `LOG_CONFIG` (cell 4), derived
sigma range from biology + pixel size (cell 6), pipeline (cell 8),
visualisation (cell 10), per-run overrides (cell 12), calibration on
one image (cell 14), batch on N images (cell 16).

**Cost.** `blob_log` at sigma 16-50 on a 2304x2304 image is ~5-15 s.
A 10-image batch is 1-3 minutes; lower `log_num_sigma` to speed up.


## 1. Imports + repo root


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / ".git").is_dir():
    REPO_ROOT = REPO_ROOT.parent
SRC_ROOT = REPO_ROOT / "src"
for p in (SRC_ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from synaptic_ssl.pseudolabels.soma_log import (
    derive_log_sigmas,
    resolve_log_cfg,
    run_log_on_image,
    visualise_log_mask,
)

plt.rcParams["figure.dpi"] = 120
print("REPO_ROOT:", REPO_ROOT)


## 2. Configuration

Three groups:

- **Hardware** -- `pixel_size_nm`, the only physical constant.
- **Biology** -- `min_soma_diameter_um` / `max_soma_diameter_um`; cell 6
  converts these to LoG sigma values (`sigma = radius / sqrt(2)`).
- **Detection knobs** -- LoG threshold, NMS overlap, optional intensity
  post-filter; final shape regularisation per detected blob.


In [ ]:
LOG_CONFIG = dict(
    # ---- Data ----------------------------------------------------------
    no_patch_root="/run/media/anokhin/WinDocuments/anokhver/thesis/data/Microscopy_no_patch/20251219",
    structural_channel=2,
    file_index=0,
    batch_size=10,

    # ---- Hardware + biology --------------------------------------------
    pixel_size_nm=107.0,
    min_soma_diameter_um=5.0,      # smallest soma to detect (3-7 typical)
    max_soma_diameter_um=18.0,     # largest soma to detect (15-25 typical)

    # ---- LoG detector --------------------------------------------------
    log_num_sigma=5,               # number of scales between min/max sigma
    log_pre_smooth=2.0,            # Gaussian on raw image before LoG (px); 0 to skip
    log_threshold=0.20,            # LoG response cutoff; raise -> fewer / clearer blobs
    log_overlap=0.5,               # NMS overlap; 1.0 keeps every blob (no NMS)
    log_exclude_border=20,         # px; reject blobs whose centre is within this of the edge

    # ---- Post-detection intensity filter (optional) --------------------
    # Keeps only blobs whose mean intensity inside the disk is >= this
    # percentile of the image. None disables the filter; common values
    # are 75-90 to drop dim false positives along bright dendrites.
    intensity_filter_pct=None,

    # ---- Shape regularisation (per detection -> 2D mask) --------------
    # LoG returns (row, col, sigma); we render each as a disk of radius
    # sqrt(2)*sigma. Optionally regularise the rendered shape.
    shape_mode="circle",           # 'circle' | 'ellipse' | 'closing' | 'convex' | 'raw'
    closing_radius=8,              # used iff shape_mode='closing'
    dilate_r=0,                    # final dilation in px; 0 keeps detector radius

    # ---- Display -------------------------------------------------------
    overlay_dim=0.45,
    blob_color=(1.0, 0.2, 0.8),    # magenta-ish; visible on grey structural
)

print(
    f"min_diameter = {LOG_CONFIG['min_soma_diameter_um']} um\n"
    f"max_diameter = {LOG_CONFIG['max_soma_diameter_um']} um\n"
    f"log_threshold = {LOG_CONFIG['log_threshold']}  "
    f"num_sigma = {LOG_CONFIG['log_num_sigma']}"
)


## 3. Derived parameters

Converts the biological diameters to LoG sigma values. The relation is
`sigma = blob_radius / sqrt(2)` for the scale-normalised LoG, and
`blob_radius = soma_diameter / 2`.


In [ ]:
# (all functions now imported from synaptic_ssl.pseudolabels.soma_log)

_d = derive_log_sigmas(LOG_CONFIG)
print(f"pixel_size = {LOG_CONFIG['pixel_size_nm']} nm/px  "
      f"({LOG_CONFIG['pixel_size_nm']/1000:.4f} um/px)")
print(f"soma radius range: "
      f"{_d['min_radius_px']:.1f} - {_d['max_radius_px']:.1f} px  "
      f"({LOG_CONFIG['min_soma_diameter_um']:.1f} - {LOG_CONFIG['max_soma_diameter_um']:.1f} um)")
print(f"LoG sigma range  : "
      f"{_d['log_min_sigma']:.2f} - {_d['log_max_sigma']:.2f}  "
      f"(num_sigma={LOG_CONFIG['log_num_sigma']})")
print(f"scales sampled   : "
      f"{[f'{s:.1f}' for s in np.linspace(_d['log_min_sigma'], _d['log_max_sigma'], LOG_CONFIG['log_num_sigma'])]}")


## 4. Pipeline

Four helpers:

- `_normalize_for_log(image)` -- p1/p99 normalisation so `log_threshold`
  is comparable across images.
- `detect_log_blobs(image, cfg)` -- runs `skimage.feature.blob_log`,
  returns `(N, 3)` array of `(row, col, sigma)` detections.
- `filter_blobs_by_intensity(blobs, image, pct)` -- optional dim-blob
  rejection: keep blobs whose mean intensity inside the disk >= `pct`
  percentile of the whole image.
- `render_blob_mask(blobs, shape, cfg)` -- dispatches on `shape_mode`
  to draw circles / ellipses / convex hulls / closed disks.
- `run_log_on_image(path, cfg)` -- wraps everything for one MIP file.


In [ ]:
# (all functions now imported from synaptic_ssl.pseudolabels.soma_log)


## 5. Visualisation

`visualise_log_mask(result, cfg, *, axes=None)` -- reusable 3-panel
figure. Pass a `(1, 3)` row of axes for the batch cell, or omit to get
a standalone figure.

Panel 1: raw structural. Panel 2: structural + raw blob circles (radii
from sigma, before intensity filter). Panel 3: final mask overlay.


In [ ]:
# (all functions now imported from synaptic_ssl.pseudolabels.soma_log)


## 6. Overrides for the calibration run

Tune knobs here without scrolling back to cell 4. Inline `^/v` notes
indicate direction of effect.


In [ ]:
# Uncomment any line to override the LOG_CONFIG default for this run.
LOG_CONFIG.update(
    # file_index=0,                    # which .npy in the folder (sorted, 0-based)

    # ---- Detector size range ------------------------------------------
    min_soma_diameter_um=20.0,        # um; ^ exclude small somas | v include smaller cells
    # max_soma_diameter_um=18.0,       # um; ^ allow bigger somas | v cap soma size
    # log_num_sigma=5,                 # ^ more scales (slower, finer) | v fewer (faster, coarser)

    # ---- Detector sensitivity -----------------------------------------
    # log_pre_smooth=2.0,              # px; ^ more smoothing (suppresses noise) | v sharper response
    log_threshold=0.20,              # ^ stricter (fewer blobs) | v more blobs (incl. dim/spurious)
    # log_overlap=0.5,                 # NMS: ^ keep more overlapping detections | v stricter NMS
    # log_exclude_border=20,           # px; ^ ignore wider edge zone | v allow edge blobs

    # ---- Post-detection intensity filter ------------------------------
    # intensity_filter_pct=None,       # None = OFF; set 75-90 to drop dim false positives
    #                                  # ^ stricter (drop more) | v looser

    # ---- Shape regularisation -----------------------------------------
    # shape_mode="circle",             # 'circle' | 'ellipse' | 'closing' | 'convex' | 'raw'
    # closing_radius=8,                # px; used iff shape_mode='closing'; ^ smoother
    # dilate_r=0,                      # px; ^ thicker boundary | v keep detector radius
)

_d = derive_log_sigmas(LOG_CONFIG)
print(
    f"file_index    = {LOG_CONFIG['file_index']}\n"
    f"diameters     = {LOG_CONFIG['min_soma_diameter_um']:.1f} - {LOG_CONFIG['max_soma_diameter_um']:.1f} um  "
    f"(sigma {_d['log_min_sigma']:.1f} - {_d['log_max_sigma']:.1f})\n"
    f"log           : threshold={LOG_CONFIG['log_threshold']}  "
    f"num_sigma={LOG_CONFIG['log_num_sigma']}  "
    f"overlap={LOG_CONFIG['log_overlap']}\n"
    f"intensity_fl  = {LOG_CONFIG['intensity_filter_pct']}\n"
    f"shape_mode    = {LOG_CONFIG['shape_mode']}  dilate_r={LOG_CONFIG['dilate_r']}"
)


## 7. Calibration -- one image

Tune `LOG_CONFIG` (cell 4) or cell 12 (overrides), then re-run this
cell until the magenta circles in panel 2 cover every visible cell
body and only cell bodies.


In [ ]:
_files = sorted(Path(LOG_CONFIG["no_patch_root"]).glob("*.npy"))
if not _files:
    raise FileNotFoundError(f"no .npy MIPs found under {LOG_CONFIG['no_patch_root']}")
_path = _files[LOG_CONFIG["file_index"]]
print(f"[{LOG_CONFIG['file_index']}/{len(_files)-1}] {_path.name}")

import time
_t0 = time.time()
_result = run_log_on_image(_path, LOG_CONFIG)
_dt = time.time() - _t0
print(f"  detect: {_dt:.1f} s")
print(
    f"  raw_blobs={_result['n_raw']} -> kept={_result['n_kept']} -> "
    f"final_ccs={_result['n_blobs_final']}  "
    f"mask_cov={_result['mask'].mean():.2%}"
)
fig, _ = visualise_log_mask(_result, LOG_CONFIG)
fig.suptitle(_path.stem[:60], y=1.02, fontsize=10)
plt.show()


## 8. Batch -- N images

Runs the same `LOG_CONFIG` on `LOG_CONFIG["batch_size"]` images. Each
image takes ~5-15 s for the LoG step at default sigmas, so a 10-image
batch is 1-3 minutes total.


In [ ]:
import time

_files = sorted(Path(LOG_CONFIG["no_patch_root"]).glob("*.npy"))
if not _files:
    raise FileNotFoundError(f"no .npy MIPs found under {LOG_CONFIG['no_patch_root']}")
_batch = _files[:int(LOG_CONFIG["batch_size"])]
print(f"LoG-soma batch: {len(_batch)} images")

_t0 = time.time()
fig, axes_grid = plt.subplots(len(_batch), 3, figsize=(15, 5 * len(_batch)))
if len(_batch) == 1:
    axes_grid = np.atleast_2d(axes_grid)

_stats = []
for i, path in enumerate(_batch):
    _ti = time.time()
    result = run_log_on_image(path, LOG_CONFIG)
    visualise_log_mask(result, LOG_CONFIG, axes=axes_grid[i], title_prefix=f"[{i}] ")
    axes_grid[i, 0].set_title(f"[{i}] {path.stem[:30]}\u2026")
    _stats.append(dict(
        idx=i, name=path.name,
        n_raw=result["n_raw"], n_kept=result["n_kept"],
        n_final=result["n_blobs_final"],
        cov=float(result["mask"].mean()),
        sec=time.time() - _ti,
    ))
    print(f"  [{i:2d}] {path.name[:50]:50s}  "
          f"raw={result['n_raw']:4d}  kept={result['n_kept']:4d}  "
          f"final={result['n_blobs_final']:3d}  "
          f"cov={result['mask'].mean():.2%}  "
          f"{_stats[-1]['sec']:.1f}s")
plt.tight_layout()
plt.show()

_n  = [s["n_final"] for s in _stats]
_co = [s["cov"]     for s in _stats]
print(
    f"\nLOG-SOMA batch summary ({len(_stats)} images, "
    f"thr={LOG_CONFIG['log_threshold']}, shape={LOG_CONFIG['shape_mode']}):\n"
    f"  n_blobs : mean={np.mean(_n):.1f}  range=[{min(_n)}, {max(_n)}]\n"
    f"  cov     : mean={np.mean(_co):.2%}  range=[{min(_co):.2%}, {max(_co):.2%}]\n"
    f"  total   : {time.time() - _t0:.1f} s"
)
